# Transformer-Based Models for Apnea Detection

This notebook trains a tabular-transformer model and selects checkpoints by **validation AUC-PR**.

Notes:
- Primary objective: maximize validation AUC-PR.
- Threshold tuning is used only for reporting F1.
- The code attempts to infer sequence structure from lagged feature names.

In [ ]:
from __future__ import annotations

import copy
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from config import dir_config
from src.data_prepatation import create_batched_splits

In [ ]:
def set_seed(seed: int = 2542) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(2542)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
processed_dir = dir_config.data.processed
parquet_files = list(Path(processed_dir).glob("*_with_metadata.parquet"))
parquet_files = [f for f in parquet_files if f.name != "agg_data_with_metadata.parquet"]

target_type = "apnea"
future_target_step = 1
max_feature_lag = 5

train_X, train_y, val_X, val_y, test_X, test_y, left_out = create_batched_splits(
    parquet_files,
    batch_size=360,
    gap_size=6,
    top_features=None,
    top_features_lag=max_feature_lag,
    target_type=target_type,
    target_future_steps=future_target_step,
    val_ratio=0.2,
    test_ratio=0.2,
    n_leave_out=5,
    random_seed=2542,
)

print("train/val/test sizes:", train_X.shape, val_X.shape, test_X.shape)
print("positive rates:", {"train": float(train_y.mean()), "val": float(val_y.mean()), "test": float(test_y.mean())})

In [ ]:
def _extract_lag(col: str) -> int:
    parts = col.split("_")
    for i in range(len(parts) - 1):
        if parts[i] == "lag":
            try:
                return int(parts[i + 1])
            except ValueError:
                return 0
    return 0


def _base_name(col: str) -> str:
    marker = "_lag_"
    if marker in col:
        return col.split(marker)[0]
    return col


def build_sequence_tensors(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, int]]:
    cols = list(train_df.columns)
    lags = np.array([_extract_lag(c) for c in cols], dtype=np.int64)
    unique_lags = sorted(np.unique(lags).tolist())

    # Group columns by base feature and lag; drop incomplete bases for a rectangular tensor
    by_base: Dict[str, Dict[int, int]] = {}
    for idx, col in enumerate(cols):
        base = _base_name(col)
        lag = int(lags[idx])
        if base not in by_base:
            by_base[base] = {}
        by_base[base][lag] = idx

    complete_bases = [b for b, m in by_base.items() if all(l in m for l in unique_lags)]
    complete_bases = sorted(complete_bases)

    if len(complete_bases) == 0:
        # Fallback: single-token sequence if no lag pattern was found
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        tr = scaler.fit_transform(imputer.fit_transform(train_df.values))
        va = scaler.transform(imputer.transform(val_df.values))
        te = scaler.transform(imputer.transform(test_df.values))
        tr = np.asarray(tr, dtype=np.float32)[:, None, :]
        va = np.asarray(va, dtype=np.float32)[:, None, :]
        te = np.asarray(te, dtype=np.float32)[:, None, :]
        meta = {"seq_len": 1, "n_features": tr.shape[-1], "n_bases": tr.shape[-1]}
        return tr, va, te, meta

    seq_len = len(unique_lags)
    n_bases = len(complete_bases)

    idx_order: List[int] = []
    for lag in unique_lags:
        for base in complete_bases:
            idx_order.append(by_base[base][lag])

    train_sel = train_df.iloc[:, idx_order].values
    val_sel = val_df.iloc[:, idx_order].values
    test_sel = test_df.iloc[:, idx_order].values

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    tr = scaler.fit_transform(imputer.fit_transform(train_sel))
    va = scaler.transform(imputer.transform(val_sel))
    te = scaler.transform(imputer.transform(test_sel))

    tr = np.asarray(tr, dtype=np.float32).reshape(len(train_df), seq_len, n_bases)
    va = np.asarray(va, dtype=np.float32).reshape(len(val_df), seq_len, n_bases)
    te = np.asarray(te, dtype=np.float32).reshape(len(test_df), seq_len, n_bases)

    meta = {"seq_len": seq_len, "n_features": n_bases, "n_bases": n_bases}
    return tr, va, te, meta

In [ ]:
train_seq, val_seq, test_seq, seq_meta = build_sequence_tensors(train_X, val_X, test_X)

train_targets = torch.tensor(train_y.to_numpy(), dtype=torch.float32).unsqueeze(1)
val_targets = torch.tensor(val_y.to_numpy(), dtype=torch.float32).unsqueeze(1)
test_targets = torch.tensor(test_y.to_numpy(), dtype=torch.float32).unsqueeze(1)

train_tensor = torch.tensor(train_seq, dtype=torch.float32)
val_tensor = torch.tensor(val_seq, dtype=torch.float32)
test_tensor = torch.tensor(test_seq, dtype=torch.float32)

print("sequence shape (N, T, F):", train_tensor.shape)
print("seq meta:", seq_meta)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        return x + self.pe[:, : x.size(1)]


class TabularTransformer(nn.Module):
    def __init__(self, input_dim: int, seq_len: int, d_model: int = 128, nhead: int = 8, num_layers: int = 3, ff_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max(16, seq_len + 2))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)
        x = self.norm(x.mean(dim=1))
        return self.head(x)

In [ ]:
def evaluate_torch_binary(model: nn.Module, X: torch.Tensor, y: torch.Tensor, split_name: str, threshold: float = 0.5) -> Dict[str, object]:
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X.to(device)).squeeze(1)).detach().cpu().numpy()
    y_true = y.squeeze(1).detach().cpu().numpy().astype(int)
    y_pred = (probs >= threshold).astype(int)
    m = {
        "auc_roc": roc_auc_score(y_true, probs),
        "auc_pr": average_precision_score(y_true, probs),
        "f1": f1_score(y_true, y_pred),
        "classification_report": classification_report(y_true, y_pred),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }
    print(f"{split_name:10s} AUC-ROC={m['auc_roc']:.4f} AUC-PR={m['auc_pr']:.4f} F1={m['f1']:.4f}")
    return m


def best_f1_threshold(y_true: np.ndarray, probs: np.ndarray) -> Tuple[float, float]:
    from sklearn.metrics import precision_recall_curve

    p, r, t = precision_recall_curve(y_true, probs)
    f1s = 2 * (p[:-1] * r[:-1]) / (p[:-1] + r[:-1] + 1e-12)
    i = int(np.argmax(f1s))
    return float(t[i]), float(f1s[i])

In [ ]:
train_ds = TensorDataset(train_tensor, train_targets)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

neg = float((train_y == 0).sum())
pos = float((train_y == 1).sum())
if pos == 0:
    raise ValueError("No positive examples in train split.")
base_pos_weight = neg / pos

candidates = [
    {"d_model": 128, "nhead": 8, "num_layers": 2, "ff_dim": 256, "dropout": 0.10, "lr": 8e-4, "weight_decay": 1e-4, "pos_w_scale": 0.35},
    {"d_model": 128, "nhead": 8, "num_layers": 3, "ff_dim": 256, "dropout": 0.15, "lr": 8e-4, "weight_decay": 2e-4, "pos_w_scale": 0.35},
    {"d_model": 192, "nhead": 8, "num_layers": 3, "ff_dim": 384, "dropout": 0.10, "lr": 6e-4, "weight_decay": 2e-4, "pos_w_scale": 0.30},
]

MAX_EPOCHS = 90
PATIENCE = 14


def fit_one(cfg: Dict[str, float]) -> Dict[str, object]:
    model = TabularTransformer(
        input_dim=train_tensor.shape[-1],
        seq_len=train_tensor.shape[1],
        d_model=cfg["d_model"],
        nhead=cfg["nhead"],
        num_layers=cfg["num_layers"],
        ff_dim=cfg["ff_dim"],
        dropout=cfg["dropout"],
    ).to(device)

    pos_weight = torch.tensor([base_pos_weight * cfg["pos_w_scale"]], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_aucpr = -1.0
    best_state = None
    no_imp = 0

    y_val_true = val_targets.squeeze(1).detach().cpu().numpy().astype(int)

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_probs = torch.sigmoid(model(val_tensor.to(device)).squeeze(1)).detach().cpu().numpy()
        val_aucpr = float(average_precision_score(y_val_true, val_probs))

        if val_aucpr > best_aucpr:
            best_aucpr = val_aucpr
            best_state = copy.deepcopy(model.state_dict())
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(model(val_tensor.to(device)).squeeze(1)).detach().cpu().numpy()

    thr, best_f1 = best_f1_threshold(y_val_true, val_probs)

    return {
        "cfg": cfg,
        "state": copy.deepcopy(model.state_dict()),
        "val_aucpr": best_aucpr,
        "val_best_f1": best_f1,
        "best_thr": thr,
    }


results = []
for i, cfg in enumerate(candidates, 1):
    out = fit_one(cfg)
    results.append(out)
    print(
        f"[{i}/{len(candidates)}] cfg={cfg} | val_AUC-PR={out['val_aucpr']:.4f} | "
        f"val_best_F1={out['val_best_f1']:.4f} | thr={out['best_thr']:.4f}"
    )

best = max(results, key=lambda x: x["val_aucpr"])
print("\nBest by validation AUC-PR:", best["cfg"])
print("Best validation AUC-PR:", round(best["val_aucpr"], 6))

best_model = TabularTransformer(
    input_dim=train_tensor.shape[-1],
    seq_len=train_tensor.shape[1],
    d_model=best["cfg"]["d_model"],
    nhead=best["cfg"]["nhead"],
    num_layers=best["cfg"]["num_layers"],
    ff_dim=best["cfg"]["ff_dim"],
    dropout=best["cfg"]["dropout"],
).to(device)
best_model.load_state_dict(best["state"])
best_threshold = best["best_thr"]

print("Selected threshold (for F1 reporting only):", round(best_threshold, 6))

train_metrics = evaluate_torch_binary(best_model, train_tensor, train_targets, "Train", threshold=best_threshold)
val_metrics = evaluate_torch_binary(best_model, val_tensor, val_targets, "Validation", threshold=best_threshold)
test_metrics = evaluate_torch_binary(best_model, test_tensor, test_targets, "Test", threshold=best_threshold)

print("\nAUC-PR summary (selection objective)")
print({
    "train_auc_pr": float(train_metrics["auc_pr"]),
    "val_auc_pr": float(val_metrics["auc_pr"]),
    "test_auc_pr": float(test_metrics["auc_pr"]),
})